# Data Selection and Cleaning

## 1. Loading the Data
The required Python libraries were imported and the 5 datasets were loaded into the notebook.  

In [295]:
import pandas as pd
import numpy as np
from functools import reduce

In [296]:
# 1. 
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

- The datasets cover different dimensions of the "Sustainable Living Quality Index", including economic, health and education, infrastructure, environment, and governance.
- The missing value symbol `..` was treated as a missing value.

In [297]:
economic_data = pd.read_csv("../Dataset/economic_indicators.csv", na_values=[".."])
infrastructure_data = pd.read_csv('../Dataset/infrastructure_indicators.csv', na_values=[".."])
environmental_data = pd.read_csv('../Dataset/environment_indicators.csv', na_values=[".."])
health_education_data = pd.read_csv('../Dataset/health_education_indicators.csv', na_values=[".."])
governance_data = pd.read_csv('../Dataset/governance_indicators.csv', na_values=[".."])


- After loading the datasets, the shape of each dataset was checked.  

In [298]:
print("Economic:", economic_data.shape)
print("Health and Education:", health_education_data.shape)
print("Infrastructure:", infrastructure_data.shape)
print("Environment:", environmental_data.shape)
print("Governance:", governance_data.shape)

Economic: (656, 10)
Health and Education: (656, 10)
Infrastructure: (656, 10)
Environment: (656, 10)
Governance: (653, 9)


## 2. Choosing Relevant Variables
relevant variables were selected from each dataset. The variables were grouped into five main dimensions:
- Economic conditions
- Health and education
- Infrastructure and basic services
- Environmental sustainability
- Governance and stability

Not all variables were selected, and only the most recent data were used.


### Economic

In [299]:
# 1. Define indicators to keep
economic_selected = economic_data[['Country Name', 'Country Code', 'Series Name','2020 [YR2020]','2021 [YR2021]','2022 [YR2022]','2023 [YR2023]','2024 [YR2024]']].copy()

# Keep only selected variables
economic_selected = economic_selected[
    economic_selected['Series Name'].isin([
        'GDP per capita (current US$)',
        'Inflation, consumer prices (annual %)',
        'Unemployment, total (% of total labor force) (modeled ILO estimate)'
    ])
].copy()

# Convert values to numeric
year_columns = ['2020 [YR2020]', '2021 [YR2021]', '2022 [YR2022]', '2023 [YR2023]', '2024 [YR2024]']
for col in year_columns:
    economic_selected[col] = pd.to_numeric(economic_selected[col], errors='coerce')

# Use 2024 first, if missing use 2023
economic_selected['selected_value'] = (economic_selected['2024 [YR2024]']
    .fillna(economic_selected['2023 [YR2023]'])
    .fillna(economic_selected['2022 [YR2022]'])
    .fillna(economic_selected['2021 [YR2021]'])
    .fillna(economic_selected['2020 [YR2020]'])
)

# Convert to wide format
economic_final = economic_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

# Rename columns
economic_final = economic_final.rename(columns={
    'Country Name': 'Country',
    'GDP per capita (current US$)': 'gdp_per_capita_recent',
    'Inflation, consumer prices (annual %)': 'inflation_recent',
    'Unemployment, total (% of total labor force) (modeled ILO estimate)': 'unemployment_recent'
})

economic_final.columns.name = None

print(economic_final.shape)
economic_final.head(211)

(215, 5)


,Country,Country Code,gdp_per_capita_recent,inflation_recent,unemployment_recent
0,Afghanistan,AFG,413.757895,-6.601186,13.687
1,Albania,ALB,11377.775743,2.215874,10.689
2,Algeria,DZA,5752.990767,4.046115,11.655
3,American Samoa,ASM,18017.458938,NaN,NaN
4,Andorra,AND,49303.649167,NaN,NaN
5,Angola,AGO,2665.874448,28.240495,14.020
6,Antigua and Barbuda,ATG,23542.452695,6.198867,NaN
7,Argentina,ARG,13969.783660,219.883929,7.150
8,Armenia,ARM,8556.214070,0.269512,12.405
9,Aruba,ABW,39498.594129,NaN,NaN


- 2024 was used as the reference year. Where 2024 values were missing, 2023 values were used as a one-year backfill.

### Health and Education

In [300]:
health_education_selected = health_education_data[['Country Name', 'Country Code', 'Series Name','2020 [YR2020]','2021 [YR2021]','2022 [YR2022]','2023 [YR2023]','2024 [YR2024]']].copy()

health_education_selected = health_education_selected[
    health_education_selected['Series Name'].isin([
        'Life expectancy at birth, total (years)',
        'School enrollment, secondary (% gross)',
    ])
].copy()

# Convert values to numeric
year_columns = ['2020 [YR2020]','2021 [YR2021]','2022 [YR2022]','2023 [YR2023]','2024 [YR2024]']

for col in year_columns:
    health_education_selected[col] = pd.to_numeric(
        health_education_selected[col],
        errors='coerce'
    )

# Create selected value column
health_education_selected['selected_value'] = np.nan

# Life expectancy: use 2024 first, if missing use 2023
life_mask = health_education_selected['Series Name'] == 'Life expectancy at birth, total (years)'
health_education_selected.loc[life_mask, 'selected_value'] = (
    health_education_selected.loc[life_mask, '2024 [YR2024]']
    .fillna(health_education_selected.loc[life_mask, '2023 [YR2023]'])
)

# Secondary enrollment: latest available from 2024 to 2020
secondary_mask = health_education_selected['Series Name'] == 'School enrollment, secondary (% gross)'
health_education_selected.loc[secondary_mask, 'selected_value'] = (
    health_education_selected.loc[secondary_mask, '2024 [YR2024]']
    .fillna(health_education_selected.loc[secondary_mask, '2023 [YR2023]'])
    .fillna(health_education_selected.loc[secondary_mask, '2022 [YR2022]'])
    .fillna(health_education_selected.loc[secondary_mask, '2021 [YR2021]'])
    .fillna(health_education_selected.loc[secondary_mask, '2020 [YR2020]'])
)

# Pivot to wide format
health_education_final = health_education_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

health_education_final = health_education_final.rename(columns={
    'Country Name': 'Country',
    'Life expectancy at birth, total (years)': 'life_expectancy_2024',
    'School enrollment, secondary (% gross)': 'secondary_enrollment_recent',
})

health_education_final.columns.name = None

print(health_education_final.shape)
health_education_final.head(217)

(217, 4)


,Country,Country Code,life_expectancy_2024,secondary_enrollment_recent
0,Afghanistan,AFG,66.289000,59.613602
1,Albania,ALB,79.776000,108.355392
2,Algeria,DZA,76.475000,105.164132
3,American Samoa,ASM,72.992000,NaN
4,Andorra,AND,84.188000,101.923820
5,Angola,AGO,64.805000,51.483905
6,Antigua and Barbuda,ATG,77.766000,108.880779
7,Argentina,ARG,77.543000,105.574584
8,Armenia,ARM,78.319512,90.945871
9,Aruba,ABW,76.500000,124.379367


### Infrastructure

In [301]:
infrastructure_selected = infrastructure_data[['Country Name', 'Country Code', 'Series Name', '2023 [YR2023]', '2024 [YR2024]']].copy()

infrastructure_selected = infrastructure_selected[
    infrastructure_selected['Series Name'].isin([
        'Access to electricity (% of population)',
        'People using at least basic drinking water services (% of population)',
        'People using at least basic sanitation services (% of population)'
    ])
].copy()

infrastructure_selected['2023 [YR2023]'] = pd.to_numeric(infrastructure_selected['2023 [YR2023]'], errors='coerce')
infrastructure_selected['2024 [YR2024]'] = pd.to_numeric(infrastructure_selected['2024 [YR2024]'], errors='coerce')

infrastructure_selected['selected_value'] = infrastructure_selected['2024 [YR2024]'].fillna(infrastructure_selected['2023 [YR2023]'])

infrastructure_final = infrastructure_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

infrastructure_final = infrastructure_final.rename(columns={
    'Country Name': 'Country',
    'Access to electricity (% of population)': 'electricity_access_2024',
    'People using at least basic drinking water services (% of population)': 'drinking_water_access',
    'People using at least basic sanitation services (% of population)': 'sanitation_access'
})

infrastructure_final.columns.name = None

print(infrastructure_final.shape)
infrastructure_final.head(216)

(216, 5)


,Country,Country Code,electricity_access_2024,drinking_water_access,sanitation_access
0,Afghanistan,AFG,85.3,80.832437,54.495799
1,Albania,ALB,100.0,95.119427,99.299715
2,Algeria,DZA,100.0,92.420029,85.906486
3,American Samoa,ASM,NaN,100.000000,62.119698
4,Andorra,AND,100.0,100.000000,100.000000
5,Angola,AGO,51.1,67.960742,NaN
6,Antigua and Barbuda,ATG,100.0,98.922727,99.605719
7,Argentina,ARG,100.0,NaN,NaN
8,Armenia,ARM,100.0,99.931572,93.664943
9,Aruba,ABW,100.0,NaN,98.840002


### Environment

In [302]:
environment_selected = environmental_data[['Country Name', 'Country Code', 'Series Name', '2020 [YR2020]', '2021 [YR2021]', '2022 [YR2022]', '2023 [YR2023]']].copy()

environment_selected = environment_selected[
    environment_selected['Series Name'].isin([
        'Forest area (% of land area)',
        'PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)',
        'Renewable energy consumption (% of total final energy consumption)'
    ])
].copy()

# Convert values to numeric
environment_selected['2020 [YR2020]'] = pd.to_numeric(environment_selected['2020 [YR2020]'], errors='coerce')
environment_selected['2021 [YR2021]'] = pd.to_numeric(environment_selected['2021 [YR2021]'], errors='coerce')
environment_selected['2022 [YR2022]'] = pd.to_numeric(environment_selected['2022 [YR2022]'], errors='coerce')
environment_selected['2023 [YR2023]'] = pd.to_numeric(environment_selected['2023 [YR2023]'], errors='coerce')

# Create selected value column
environment_selected['selected_value'] = np.nan

# Forest area: use 2023 first, if missing use 2022
forest_mask = environment_selected['Series Name'] == 'Forest area (% of land area)'
environment_selected.loc[forest_mask, 'selected_value'] = (
    environment_selected.loc[forest_mask, '2023 [YR2023]']
    .fillna(environment_selected.loc[forest_mask, '2022 [YR2022]'])
)

# PM2.5: use 2020 because only 2020 data is available for most countries.
pm25_mask = environment_selected['Series Name'] == 'PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)'
environment_selected.loc[pm25_mask, 'selected_value'] = environment_selected.loc[pm25_mask, '2020 [YR2020]']

# Renewable energy: use 2022 first, if missing use 2021
renewable_mask = environment_selected['Series Name'] == 'Renewable energy consumption (% of total final energy consumption)'
environment_selected.loc[renewable_mask, 'selected_value'] = (
    environment_selected.loc[renewable_mask, '2022 [YR2022]']
    .fillna(environment_selected.loc[renewable_mask, '2021 [YR2021]'])
)

environment_final = environment_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

environment_final = environment_final.rename(columns={
    'Country Name': 'Country',
    'Forest area (% of land area)': 'forest_area_2023_reference',
    'PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)': 'pm2.5_exposure_2020',
    'Renewable energy consumption (% of total final energy consumption)': 'renewable_energy_recent'
})

environment_final.columns.name = None

print(environment_final.shape)
environment_final.head(215)

(215, 5)


,Country,Country Code,forest_area_2023_reference,pm2.5_exposure_2020,renewable_energy_recent
0,Afghanistan,AFG,1.852782,46.087094,20.0
1,Albania,ALB,28.791971,15.707004,41.9
2,Algeria,DZA,0.830314,25.552656,0.1
3,American Samoa,ASM,85.200000,6.715147,0.4
4,Andorra,AND,34.042553,9.080281,18.7
5,Angola,AGO,52.091270,25.145238,52.9
6,Antigua and Barbuda,ATG,18.013409,19.698270,0.9
7,Argentina,ARG,10.321295,14.908174,9.2
8,Armenia,ARM,11.618316,30.579633,9.1
9,Aruba,ABW,2.333333,NaN,8.8


### Governance

In [303]:
governance_selected = governance_data[['Country Name', 'Country Code', 'Series Name', '2023 [YR2023]', '2024 [YR2024]']].copy()

governance_selected = governance_selected[
    governance_selected['Series Name'].isin([
        'Government Effectiveness - Governance estimate (approx. -2.5 to +2.5)',
        'Rule of Law - Governance estimate (approx. -2.5 to +2.5)',
        'Political Stability - Governance estimate (approx. -2.5 to +2.5)'
    ])
].copy()

# Convert values to numeric
governance_selected['2023 [YR2023]'] = pd.to_numeric(governance_selected['2023 [YR2023]'], errors='coerce')
governance_selected['2024 [YR2024]'] = pd.to_numeric(governance_selected['2024 [YR2024]'], errors='coerce')
governance_selected['selected_value'] = governance_selected['2024 [YR2024]'].fillna(governance_selected['2023 [YR2023]'])

governance_final = governance_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

governance_final = governance_final.rename(columns={
    'Country Name': 'Country',
    'Government Effectiveness - Governance estimate (approx. -2.5 to +2.5)': 'government_effectiveness_2024',
    'Rule of Law - Governance estimate (approx. -2.5 to +2.5)': 'rule_of_law_2024',
    'Political Stability - Governance estimate (approx. -2.5 to +2.5)': 'political_stability_2024'
})

governance_final.columns.name = None

print(governance_final.shape)
governance_final.head()

(215, 5)


,Country,Country Code,government_effectiveness_2024,political_stability_2024,rule_of_law_2024
0,Afghanistan,AFG,-1.933997,-2.205211,-1.935489
1,Albania,ALB,0.312344,0.020467,-0.118502
2,Algeria,DZA,-0.251191,-0.693588,-0.696777
3,American Samoa,ASM,0.767382,1.253029,1.086581
4,Andorra,AND,1.159037,1.543796,1.409905


## 3. Combining the Datasets 
Economic / Environment / Governance / Health & Education / Infrastructure indicators.

After selecting and cleaning the relevant variables from each dataset, the five cleaned datasets were combined into one final dataset. The datasets were stored in a list and merged using the `reduce()` function. `Country` and `Country Code` were used as the common keys for merging. 

In [304]:
# Combine datasets

dfs = [economic_final, health_education_final, infrastructure_final, governance_final, environment_final]

merge_dfs = lambda left, right: pd.merge(left, right, on=['Country', 'Country Code'], how='outer')

final_dataset_SLQI = reduce(merge_dfs, dfs)

print(final_dataset_SLQI.shape)
final_dataset_SLQI.head()

(225, 16)


,Country,Country Code,gdp_per_capita_recent,inflation_recent,unemployment_recent,life_expectancy_2024,secondary_enrollment_recent,electricity_access_2024,drinking_water_access,sanitation_access,government_effectiveness_2024,political_stability_2024,rule_of_law_2024,forest_area_2023_reference,pm2.5_exposure_2020,renewable_energy_recent
0,Afghanistan,AFG,413.757895,-6.601186,13.687,66.289,59.613602,85.3,80.832437,54.495799,-1.933997,-2.205211,-1.935489,1.852782,46.087094,20.0
1,Albania,ALB,11377.775743,2.215874,10.689,79.776,108.355392,100.0,95.119427,99.299715,0.312344,0.020467,-0.118502,28.791971,15.707004,41.9
2,Algeria,DZA,5752.990767,4.046115,11.655,76.475,105.164132,100.0,92.420029,85.906486,-0.251191,-0.693588,-0.696777,0.830314,25.552656,0.1
3,American Samoa,ASM,18017.458938,NaN,NaN,72.992,NaN,NaN,100.000000,62.119698,0.767382,1.253029,1.086581,85.200000,6.715147,0.4
4,Andorra,AND,49303.649167,NaN,NaN,84.188,101.923820,100.0,100.000000,100.000000,1.159037,1.543796,1.409905,34.042553,9.080281,18.7


In [305]:
# Rename columns for consistency

final_dataset_SLQI = final_dataset_SLQI.rename(columns={
    'gdp_per_capita_recent': 'gdp_per_capita',
    'inflation_recent': 'inflation',
    'unemployment_recent': 'unemployment_rate',
    
    'life_expectancy_2024': 'life_expectancy',
    'secondary_enrollment_recent': 'secondary_enrollment',
    
    'electricity_access_2024': 'electricity_access',
    
    'government_effectiveness_2024': 'government_effectiveness',
    'political_stability_2024': 'political_stability',
    'rule_of_law_2024': 'rule_of_law',
    
    'forest_area_2023_reference': 'forest_area',
    'pm2.5_exposure_2020': 'pm2.5_exposure',
    'renewable_energy_recent': 'renewable_energy'
})

final_dataset_SLQI.head()

,Country,Country Code,gdp_per_capita,inflation,unemployment_rate,life_expectancy,secondary_enrollment,electricity_access,drinking_water_access,sanitation_access,government_effectiveness,political_stability,rule_of_law,forest_area,pm2.5_exposure,renewable_energy
0,Afghanistan,AFG,413.757895,-6.601186,13.687,66.289,59.613602,85.3,80.832437,54.495799,-1.933997,-2.205211,-1.935489,1.852782,46.087094,20.0
1,Albania,ALB,11377.775743,2.215874,10.689,79.776,108.355392,100.0,95.119427,99.299715,0.312344,0.020467,-0.118502,28.791971,15.707004,41.9
2,Algeria,DZA,5752.990767,4.046115,11.655,76.475,105.164132,100.0,92.420029,85.906486,-0.251191,-0.693588,-0.696777,0.830314,25.552656,0.1
3,American Samoa,ASM,18017.458938,NaN,NaN,72.992,NaN,NaN,100.000000,62.119698,0.767382,1.253029,1.086581,85.200000,6.715147,0.4
4,Andorra,AND,49303.649167,NaN,NaN,84.188,101.923820,100.0,100.000000,100.000000,1.159037,1.543796,1.409905,34.042553,9.080281,18.7


The column names were renamed into a consistent format using short and readable variable names.  
Year labels were removed from the final column names because some indicators used the latest available values rather than only one fixed year.  

## 4. Check missing values in final dataset

In [306]:
print("Missing values in final dataset:")
print(final_dataset_SLQI.isnull().sum())

print("\nFinal dataset shape:")
print(final_dataset_SLQI.shape)

Missing values in final dataset:
Country                      0
Country Code                 0
gdp_per_capita              14
inflation                   46
unemployment_rate           38
life_expectancy              8
secondary_enrollment        49
electricity_access          10
drinking_water_access       22
sanitation_access           29
government_effectiveness    12
political_stability         10
rule_of_law                 10
forest_area                 12
pm2.5_exposure              25
renewable_energy            13
dtype: int64

Final dataset shape:
(225, 16)


- Checked missing values in each column
- checked to understand the number of countries and variables after merging.  

In [307]:
print("\nData types:")
print(final_dataset_SLQI.dtypes)


Data types:
Country                         str
Country Code                    str
gdp_per_capita              float64
inflation                   float64
unemployment_rate           float64
life_expectancy             float64
secondary_enrollment        float64
electricity_access          float64
drinking_water_access       float64
sanitation_access           float64
government_effectiveness    float64
political_stability         float64
rule_of_law                 float64
forest_area                 float64
pm2.5_exposure              float64
renewable_energy            float64
dtype: object


- checked to make sure that indicator columns were stored as numeric values.

In [308]:
# Count missing values for each country
indicator_columns = final_dataset_SLQI.columns.drop(['Country', 'Country Code'])

final_dataset_SLQI['missing_values_count'] = final_dataset_SLQI[indicator_columns].isnull().sum(axis=1)
final_dataset_SLQI['available_values_count'] = final_dataset_SLQI[indicator_columns].notnull().sum(axis=1)

final_dataset_SLQI[['Country', 'Country Code', 'missing_values_count', 'available_values_count']].head(225)

,Country,Country Code,missing_values_count,available_values_count
0,Afghanistan,AFG,0,14
1,Albania,ALB,0,14
2,Algeria,DZA,0,14
3,American Samoa,ASM,4,10
4,Andorra,AND,2,12
5,Angola,AGO,1,13
6,Anguilla,AIA,11,3
7,Antigua and Barbuda,ATG,1,13
8,Argentina,ARG,2,12
9,Armenia,ARM,0,14


In [309]:
# Check missing percentage
indicator_columns = final_dataset_SLQI.columns.drop(['Country', 'Country Code', 'missing_values_count', 'available_values_count'],errors='ignore')

# Calculate missing percentage for each variable, will remove it if over 25%.
missing_summary = pd.DataFrame({
    'missing_count': final_dataset_SLQI[indicator_columns].isnull().sum(),
    'missing_percentage': final_dataset_SLQI[indicator_columns].isnull().sum() / len(final_dataset_SLQI) * 100
})

missing_summary = missing_summary.sort_values(by='missing_percentage', ascending=False)
missing_summary

,missing_count,missing_percentage
secondary_enrollment,49,21.777778
inflation,46,20.444444
unemployment_rate,38,16.888889
sanitation_access,29,12.888889
pm2.5_exposure,25,11.111111
drinking_water_access,22,9.777778
gdp_per_capita,14,6.222222
renewable_energy,13,5.777778
government_effectiveness,12,5.333333
forest_area,12,5.333333


- The number and percentage of missing values were calculated for each variable.  It helps to identify which indicators have the most missing data and may need further cleaning or removal.
- Tertiary enrolment was removed because it contained a high number of missing values even after using the latest available data from 2020 to 2024. (70 missing values and over 30%)

In [310]:
# Recalculate missing values for each country
indicator_columns = final_dataset_SLQI.columns.drop(['Country', 'Country Code', 'missing_values_count', 'available_values_count'],errors='ignore')

final_dataset_SLQI['missing_values_count'] = final_dataset_SLQI[indicator_columns].isnull().sum(axis=1)
final_dataset_SLQI['available_values_count'] = final_dataset_SLQI[indicator_columns].notnull().sum(axis=1)

final_dataset_SLQI[['Country', 'Country Code', 'missing_values_count', 'available_values_count']].sort_values(by='missing_values_count', ascending=False).head(100)

,Country,Country Code,missing_values_count,available_values_count
102,Jersey,JEY,11,3
129,Martinique,MTQ,11,3
47,Cook Islands,COK,11,3
197,"Taiwan, China",TWN,11,3
166,Reunion,REU,11,3
150,Niue,NIU,11,3
71,French Guiana,GUF,11,3
6,Anguilla,AIA,11,3
40,Channel Islands,CHI,10,4
179,Sint Maarten (Dutch part),SXM,9,5


- I also checked missing values at the country level. These countries may need to be removed later if there is not enough data to calculate a reliable index score.

In [311]:
# Remove countries with too many missing values
max_missing_allowed = 4
final_dataset_clean = final_dataset_SLQI[final_dataset_SLQI['missing_values_count'] <= max_missing_allowed].copy()

print(f"Original countries: {len(final_dataset_SLQI)}")
print(f"Countries after cleaning: {len(final_dataset_clean)}")

final_dataset_clean.head(206)

Original countries: 225
Countries after cleaning: 206


,Country,Country Code,gdp_per_capita,inflation,unemployment_rate,life_expectancy,secondary_enrollment,electricity_access,drinking_water_access,sanitation_access,government_effectiveness,political_stability,rule_of_law,forest_area,pm2.5_exposure,renewable_energy,missing_values_count,available_values_count
0,Afghanistan,AFG,413.757895,-6.601186,13.687,66.289000,59.613602,85.3,80.832437,54.495799,-1.933997,-2.205211,-1.935489,1.852782,46.087094,20.0,0,14
1,Albania,ALB,11377.775743,2.215874,10.689,79.776000,108.355392,100.0,95.119427,99.299715,0.312344,0.020467,-0.118502,28.791971,15.707004,41.9,0,14
2,Algeria,DZA,5752.990767,4.046115,11.655,76.475000,105.164132,100.0,92.420029,85.906486,-0.251191,-0.693588,-0.696777,0.830314,25.552656,0.1,0,14
3,American Samoa,ASM,18017.458938,NaN,NaN,72.992000,NaN,NaN,100.000000,62.119698,0.767382,1.253029,1.086581,85.200000,6.715147,0.4,4,10
4,Andorra,AND,49303.649167,NaN,NaN,84.188000,101.923820,100.0,100.000000,100.000000,1.159037,1.543796,1.409905,34.042553,9.080281,18.7,2,12
5,Angola,AGO,2665.874448,28.240495,14.020,64.805000,51.483905,51.1,67.960742,NaN,-0.785191,-0.573596,-1.169741,52.091270,25.145238,52.9,1,13
7,Antigua and Barbuda,ATG,23542.452695,6.198867,NaN,77.766000,108.880779,100.0,98.922727,99.605719,0.429970,1.073149,0.601773,18.013409,19.698270,0.9,1,13
8,Argentina,ARG,13969.783660,219.883929,7.150,77.543000,105.574584,100.0,NaN,NaN,0.183748,-0.169746,-0.251380,10.321295,14.908174,9.2,2,12
9,Armenia,ARM,8556.214070,0.269512,12.405,78.319512,90.945871,100.0,99.931572,93.664943,-0.364029,-0.723297,-0.142394,11.618316,30.579633,9.1,0,14
10,Aruba,ABW,39498.594129,NaN,NaN,76.500000,124.379367,100.0,NaN,98.840002,0.656786,1.330210,1.220060,2.333333,NaN,8.8,4,10


- Countries with more than 4 missing indicators were excluded. 

## 5. Handle Missing Value

After checking the missing values in the final combined dataset, missing values were handled in two stages.

- First, countries with too many missing indicators were removed because they did not have enough data to calculate a reliable Sustainable Living Quality Index.

- Second, the remaining small number of missing values were filled using median imputation.

In [312]:
print("Missing values after removing countries with too much missing data:")
print(final_dataset_clean.isnull().sum())

Missing values after removing countries with too much missing data:
Country                      0
Country Code                 0
gdp_per_capita               4
inflation                   28
unemployment_rate           20
life_expectancy              0
secondary_enrollment        34
electricity_access           1
drinking_water_access       11
sanitation_access           15
government_effectiveness     2
political_stability          0
rule_of_law                  0
forest_area                  2
pm2.5_exposure               7
renewable_energy             2
missing_values_count         0
available_values_count       0
dtype: int64


In [ ]:
# # Fill remaining missing values with median
indicator_columns = final_dataset_clean.columns.drop( ['Country', 'Country Code', 'missing_values_count', 'available_values_count'],errors='ignore')

for col in indicator_columns:
    final_dataset_clean[col] = final_dataset_clean[col].fillna(
        final_dataset_clean[col].median()
    )

print("Missing values after median imputation:")
print(final_dataset_clean.isnull().sum())

Missing values after median imputation:
Country                     0
Country Code                0
gdp_per_capita              0
inflation                   0
unemployment_rate           0
life_expectancy             0
secondary_enrollment        0
electricity_access          0
drinking_water_access       0
sanitation_access           0
government_effectiveness    0
political_stability         0
rule_of_law                 0
forest_area                 0
pm2.5_exposure              0
renewable_energy            0
dtype: int64


- The remaining small number of missing values were filled using median imputation.

In [321]:
final_dataset_clean = final_dataset_clean.drop(columns=['missing_values_count', 'available_values_count'], errors='ignore')
print(f"Final dataset size: {len(final_dataset_clean)}")
final_dataset_clean.head(206)

Final dataset size: 206


,Country,Country Code,gdp_per_capita,inflation,unemployment_rate,life_expectancy,secondary_enrollment,electricity_access,drinking_water_access,sanitation_access,government_effectiveness,political_stability,rule_of_law,forest_area,pm2.5_exposure,renewable_energy
0,Afghanistan,AFG,413.757895,-6.601186,13.687000,66.289000,59.613602,85.300000,80.832437,54.495799,-1.933997,-2.205211,-1.935489,1.852782,46.087094,20.000000
1,Albania,ALB,11377.775743,2.215874,10.689000,79.776000,108.355392,100.000000,95.119427,99.299715,0.312344,0.020467,-0.118502,28.791971,15.707004,41.900000
2,Algeria,DZA,5752.990767,4.046115,11.655000,76.475000,105.164132,100.000000,92.420029,85.906486,-0.251191,-0.693588,-0.696777,0.830314,25.552656,0.100000
3,American Samoa,ASM,18017.458938,8.805920,6.910833,72.992000,96.812266,92.055568,100.000000,62.119698,0.767382,1.253029,1.086581,85.200000,6.715147,0.400000
4,Andorra,AND,49303.649167,4.342366,5.585877,84.188000,101.923820,100.000000,100.000000,100.000000,1.159037,1.543796,1.409905,34.042553,9.080281,18.700000
5,Angola,AGO,2665.874448,28.240495,14.020000,64.805000,51.483905,51.100000,67.960742,44.070734,-0.785191,-0.573596,-1.169741,52.091270,25.145238,52.900000
7,Antigua and Barbuda,ATG,23542.452695,6.198867,6.691747,77.766000,108.880779,100.000000,98.922727,99.605719,0.429970,1.073149,0.601773,18.013409,19.698270,0.900000
8,Argentina,ARG,13969.783660,219.883929,7.150000,77.543000,105.574584,100.000000,87.780681,101.827827,0.183748,-0.169746,-0.251380,10.321295,14.908174,9.200000
9,Armenia,ARM,8556.214070,0.269512,12.405000,78.319512,90.945871,100.000000,99.931572,93.664943,-0.364029,-0.723297,-0.142394,11.618316,30.579633,9.100000
10,Aruba,ABW,39498.594129,5.743790,6.014615,76.500000,124.379367,100.000000,99.012338,98.840002,0.656786,1.330210,1.220060,2.333333,19.975448,8.800000
